In [1]:
import pandas as pd
import os

In [2]:
df = pd.read_excel('./nathan_to_fix.xlsx')

In [3]:
images = df.target_filename.dropna().unique()

In [4]:
import dropbox
import os
import numpy as np

In [45]:
# token = os.getenv('dropbox_access_token')
# dbx = dropbox.Dropbox(token)

# base_path = 'Nanonets/analysis'
# os.makedirs(f"{base_path}", exist_ok=True)

# images = [
#     x for x in file["target_filename"].unique()
#     if x is not None and x == x
# ]

# created_folders = set()

# def get_folder(image_name):
#     return image_name.split('_')[2]

# for image in images:
#     folder = get_folder(image)
#     if folder not in created_folders:
#         os.makedirs(f"{base_path}/{folder}", exist_ok=True)
#         created_folders.add(folder)
#     try:
#         dropbox_path = f"/Ireland Maps/{base_path}/{folder}/{image}"
#         local_path = f"{base_path}/{folder}/{image}"
#         dbx.files_download_to_file(local_path, dropbox_path)
#     except Exception as e:
#         print(f"Error downloading {image}: {e}")


In [49]:
'''
task: identify 1-page townland images
one_page_townlands = a dictionary with the townland key as the key and the set of image names as the value
res = a deque of all images names
for every row in df:
    - create a townland key: (county, barony, parish, townland, county_gv, barony_gv, parish_gv, townland_gv)
    - if the townland key is not in the dictionary, add it and set the value to the df[target_filename]
    - if the townland key is in the dictionary, add the df[target_filename] to the set of values for the townland key

for every key in one_page_townlands:
    - if the length of the set of values for the key is > 1, remove the df[target_filename] from the deque

- save res to a json file
'''

from collections import defaultdict
import json

# Build townland -> images mapping
townland_to_images = defaultdict(set)

for _, row in df.iterrows():
    townland_key = (

        row["county"],
        row["barony"],
        row["parish"],
        row["townland"],
    )

    townland_to_images[townland_key].add(row["target_filename"])

rows = []
for key, images in townland_to_images.items():
    # Convert tuple key to a string representation
    key_str = str(key)  # or str(key) if you prefer
    for image in images:
        rows.append([key_str, image])

# Create a DataFrame
output_df = pd.DataFrame(rows, columns=["Townland Key", "Page"])
output_df.sort_values(by="Page", inplace=True)
# Write to Excel
output_df.to_excel("townland_images.xlsx", index=False)


# Identify all images involved in multi-page townlands
bad_images = set()

for images in townland_to_images.values():
    if len(images) > 1:
        bad_images.update(images)

# All images in the dataset
all_images = set(df["target_filename"].dropna().unique())

print(len(all_images))

#  Keep only clean 1-page townland images
res = sorted(all_images - bad_images)
print(len(res))

# Save result to JSON
with open("one_page_townland_images.json", "w") as f:
    json.dump(res, f, indent=2)

1904
653


In [6]:
df.head()

,filename,county,barony,parish,townland,county_gv,barony_gv,parish_gv,townland_gv,adm,check_townland,check_page,sum_land_val_num,total_land_val_num,sum_total_val_num,total_total_val_num,folder,page,target_filename
0,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNAFF,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNAFF,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN CARNAFF",0,1,196.80,177.50,224.00,224.00,4.0,65.0,IRE_GRIFF_004_065.jpg
1,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNCOGGY,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,CARNCOGGY,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN CARNCOGGY",0,1,144.50,146.50,173.75,173.75,4.0,65.0,IRE_GRIFF_004_065.jpg
2,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DEEPSTOWN,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DEEPSTOWN,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN DEEPSTOWN",1,1,82.50,82.50,86.75,6.75,4.0,65.0,IRE_GRIFF_004_065.jpg
3,IRE_GRIFF_004_065.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DERRYKEIGHAN,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,DERRYKEIGHAN,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN DERRYKEIGHAN",0,1,166.15,135.90,185.05,170.55,4.0,65.0,IRE_GRIFF_004_065.jpg
4,IRE_GRIFF_004_067.csv,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,BALLYDIVITY,ANTRIM,"DUNLUCE, LOWER",DERRYKEIGHAN,BALLYDIVITY,"ANTRIM DUNLUCE, LOWER DERRYKEIGHAN BALLYDIVITY",1,1,86.15,210.65,279.50,282.50,4.0,67.0,IRE_GRIFF_004_067.jpg


In [17]:
for image in images:
    townlands_to_check = df[(df['target_filename'] == image) & (df['check_townland'] == 1)].townland.unique()
    print(townlands_to_check)
    # res = subset.filter(subset['check_townland'] == 1).townland.unique()
    # print(res)
    break

['DEEPSTOWN']


In [4]:
import os
import random
import json


In [5]:
random_images = []
for i in range(30):
    # randomly pick a folder
    folders = os.listdir('./Nanonets/analysis')
    random.shuffle(folders)
    folders.remove('.DS_Store')
    folder = folders[0]
    # pick a random file in the folder
    images = os.listdir(f'./Nanonets/analysis/{folder}')
    random.shuffle(images)
    image = images[0]
    random_images.append(str(image))

json.dumps(random_images)
with open('random_images.json', 'w') as f:
    json.dump(random_images, f)





In [2]:
import json
with open("random_images.json", "r") as f:
        one_page_townland_images = json.load(f)
len(one_page_townland_images)

13

In [2]:
import pandas as pd
df = pd.read_excel('./nathan_to_fix.xlsx')
df[df['target_filename'] == 'IRE_GRIFF_299_172.jpg']['townland'].unique().tolist()


['CLOGHVOOLIA NORTH',
 'CLOGHVOOLIA, SOUTH',
 'CURRAGHAWADDRA',
 'GLANNAGEAR',
 'KILLISSANE']

In [ ]:
2026-01-24 12:23:09,870 - INFO - Checking page IRE_GRIFF_299_172.jpg
2026-01-24 12:23:09,870 - INFO - using claude
2026-01-24 12:24:39,129 - INFO - Token usage - Input: 3339, Output: 9210, Total: 12549
2026-01-24 12:24:39,187 - INFO - Number of townlands in df_llm is equal to the number of townlands in df
2026-01-24 12:24:39,191 - INFO - using openai
2026-01-24 12:28:41,271 - INFO - Token usage - Input: 2698, Output: 7130, Total: 9828
2026-01-24 12:28:41,357 - ERROR - Number of townlands in df_llm (4) is not equal to the number of townlands in df (5) for IRE_GRIFF_299_172.jpg and openai
2026-01-24 12:28:41,358 - ERROR - df_llm: ['CLOGHVICOLLA', 'CURRAGHWADDIA', 'GLANXAGEAR', 'KILLISSANE']
2026-01-24 12:28:41,359 - ERROR - df: ['CLOGHVOOLIA NORTH', 'CLOGHVOOLIA, SOUTH', 'CURRAGHAWADDRA', 'GLANNAGEAR', 'KILLISSANE']
2026-01-24 12:28:41,359 - INFO - using gemini
2026-01-24 12:29:26,153 - INFO - Token usage - Input: 5013, Output: 8192, Total: 13205
2026-01-24 12:29:26,165 - WARNING - JSON parse error in code block: Expecting ',' delimiter: line 770 column 18 (char 29546)
2026-01-24 12:29:26,200 - ERROR - Number of townlands in df_llm (3) is not equal to the number of townlands in df (5) for IRE_GRIFF_299_172.jpg and gemini
2026-01-24 12:29:26,200 - ERROR - df_llm: ['CLOGHYOOLIA', 'CURRAGIHAWAD-', 'GLANNAGEAR.']
2026-01-24 12:29:26,201 - ERROR - df: ['CLOGHVOOLIA NORTH', 'CLOGHVOOLIA, SOUTH', 'CURRAGHAWADDRA', 'GLANNAGEAR', 'KILLISSANE']
2026-01-24 12:29:26,201 - INFO - Checking page IRE_GRIFF_142_102.jpg

In [ ]:
2026-01-24 12:45:04,323 - INFO - Checking page IRE_GRIFF_299_172.jpg
2026-01-24 12:45:04,323 - INFO - using claude
2026-01-24 12:46:33,785 - INFO - Token usage - Input: 3339, Output: 9201, Total: 12540
2026-01-24 12:46:33,861 - INFO - Number of townlands in df_llm is equal to the number of townlands in df
2026-01-24 12:46:33,865 - INFO - using openai
2026-01-24 12:49:39,209 - INFO - Token usage - Input: 2698, Output: 6171, Total: 8869
2026-01-24 12:49:39,270 - ERROR - Number of townlands in df_llm (4) is not equal to the number of townlands in df (5) for IRE_GRIFF_299_172.jpg and openai
2026-01-24 12:49:39,271 - ERROR - df_llm: ['CLOGHVICOLLA', 'CURRAGHWADDIA', 'GLANXAGEAR', 'KILLISSANE']
2026-01-24 12:49:39,271 - ERROR - df: ['CLOGHVOOLIA NORTH', 'CLOGHVOOLIA, SOUTH', 'CURRAGHAWADDRA', 'GLANNAGEAR', 'KILLISSANE']
2026-01-24 12:49:39,271 - INFO - using gemini
2026-01-24 12:50:19,678 - INFO - Token usage - Input: 5013, Output: 8192, Total: 13205
2026-01-24 12:50:19,697 - WARNING - JSON parse error in code block: Expecting ',' delimiter: line 770 column 18 (char 29557)
2026-01-24 12:50:19,739 - ERROR - Number of townlands in df_llm (3) is not equal to the number of townlands in df (5) for IRE_GRIFF_299_172.jpg and gemini
2026-01-24 12:50:19,739 - ERROR - df_llm: ['CLOGHYOOLIA', 'CURRAGIHAWAD-', 'GLANNAGEAR.']
2026-01-24 12:50:19,740 - ERROR - df: ['CLOGHVOOLIA NORTH', 'CLOGHVOOLIA, SOUTH', 'CURRAGHAWADDRA', 'GLANNAGEAR', 'KILLISSANE']
(base) nathanbehailu@Nathans-MacBook-Air ireland %   

In [2]:
# similaruty check
s1 = 'ST. FRANCIS-ABBEY'
s2 = 'ST. FRANCISABBEY'

from difflib import SequenceMatcher

# Calculate similarity ratio
score = SequenceMatcher(None, s1.upper(), s2.upper()).ratio()
print(f"Similarity ratio: {score*100:.1f}%")



Similarity ratio: 97.0%


In [11]:
import pandas as pd
df = pd.read_excel('./nathan_to_fix.xlsx')
display(df[df['target_filename'] == 'IRE_GRIFF_260_170.jpg'])


,filename,county,barony,parish,townland,county_gv,barony_gv,parish_gv,townland_gv,adm,check_townland,check_page,sum_land_val_num,total_land_val_num,sum_total_val_num,total_total_val_num,folder,page,target_filename
4578,IRE_GRIFF_260_170.csv,TIPPERARY,MIDDLETHIRD,ST. JOHN BAPTIST,ST. FRANCISABBEY,TIPPERARY,MIDDLETHIRD,ST JOHN BAPTIST,CASHEL,TIPPERARY MIDDLETHIRD ST JOHN BAPTIST CASHEL,1,1,899.10,31.25,1687.30,172.15,260.0,170.0,IRE_GRIFF_260_170.jpg
4579,IRE_GRIFF_260_170.csv,TIPPERARY,MIDDLETHIRD,ST. JOHN BAPTIST,STONEPARK,TIPPERARY,MIDDLETHIRD,ST JOHN BAPTIST,STONEPARK,TIPPERARY MIDDLETHIRD ST JOHN BAPTIST STONEPARK,0,1,34.00,34.00,34.90,34.90,260.0,170.0,IRE_GRIFF_260_170.jpg
4580,IRE_GRIFF_260_170.csv,TIPPERARY,MIDDLETHIRD,ST. JOHN BAPTIST,TOWN OF CASHEL,TIPPERARY,MIDDLETHIRD,ST JOHN BAPTIST,CASHEL,TIPPERARY MIDDLETHIRD ST JOHN BAPTIST CASHEL,1,1,899.10,31.25,1687.30,172.15,260.0,170.0,IRE_GRIFF_260_170.jpg
4581,IRE_GRIFF_260_170.csv,TIPPERARY,MIDDLETHIRD,ST. JOHN BAPTIST,WALLER'S-LOT,TIPPERARY,MIDDLETHIRD,ST JOHN BAPTIST,WALLER'S LOT,TIPPERARY MIDDLETHIRD ST JOHN BAPTIST WALLER'S...,1,1,314.65,314.65,156.35,339.45,260.0,170.0,IRE_GRIFF_260_170.jpg


In [ ]:
["IRE_GRIFF_204_112.jpg", "IRE_GRIFF_134_029.jpg", "IRE_GRIFF_236_018.jpg", "IRE_GRIFF_196_008.jpg", "IRE_GRIFF_231_134.jpg", "IRE_GRIFF_263_075.jpg", "IRE_GRIFF_225_112.jpg", "IRE_GRIFF_131_038.jpg", "IRE_GRIFF_262_089.jpg", "IRE_GRIFF_136_019.jpg", "IRE_GRIFF_139_017.jpg", "IRE_GRIFF_228_157.jpg", "IRE_GRIFF_130_035.jpg", "IRE_GRIFF_079_049.jpg", "IRE_GRIFF_142_087.jpg", "IRE_GRIFF_024_059.jpg", "IRE_GRIFF_018_430.jpg", "IRE_GRIFF_085_063.jpg", "IRE_GRIFF_130_053.jpg", "IRE_GRIFF_134_031.jpg", "IRE_GRIFF_137_146.jpg", "IRE_GRIFF_046_075.jpg", "IRE_GRIFF_135_011.jpg", "IRE_GRIFF_223_127.jpg", "IRE_GRIFF_137_146.jpg", "IRE_GRIFF_299_165.jpg", "IRE_GRIFF_239_045.jpg", "IRE_GRIFF_135_019.jpg", "IRE_GRIFF_315_013.jpg", "IRE_GRIFF_197_017.jpg"]